# Задание 4. Продажи и остатки

Использованы схемы таблиц dwh.products, dwh.sales и dwh.stock из условия.

Написать SQL-запросы на основе описанных таблиц в Clickhouse
1) Для каждого товара вывести следующую информацию
- код товара
- название товара
- бренд
- категорию
- сумму продаж в шт, за последний полный месяц
- сумму продаж в рублях, за последний полный месяц 
- среднюю цену на товара, за последний полный месяц
- средний объем продаж в штуках за последние 3 месяца (включая последний месяц)
- разницу между текущим остатком и средним объемом продаж в штуках за последние 3 месяца
В итоговом результате должны быть только товары, у которых были продажи за последний месяц.
Прошлый месяц должен прописываться в коде, нахождением этого месяца, а не "2025-05-01"
2) Вывести топ 5 брендов  по объему продаж в рублях за последний месяц, сравнить с объемом продаж за предыдущий месяц.

In [ ]:
import clickhouse_connect
import pandas as pd

# для выполнения запросов укажите параметры подключения к ClickHouse
client = clickhouse_connect.get_client(host=, port=, username=, password=) 
print("Успешное подключение!")

## Товарные показатели

In [ ]:
query_1 = """
WITH
    toStartOfMonth(today()) AS start_curr_month
    , addMonths(start_curr_month, -1) AS start_last_month
    , addMonths(start_curr_month, -3) AS start_3m_ago
    
    , sales_data AS (
        SELECT
            itemid
            , sumIf(qty, date >= start_last_month AND date < start_curr_month) AS sales_qty_last_month
            , sumIf(qty * price, date >= start_last_month AND date < start_curr_month) AS sales_rub_last_month
            , sum(qty) / 3 AS avg_qty_3m
        FROM dwh.sales
        PREWHERE date >= start_3m_ago 
            AND date < start_curr_month
        GROUP BY itemid
        HAVING sales_qty_last_month > 0
    )
    
    , stock_data AS (
        SELECT
            itemid
            , argMax(qty, date) AS actual_qty
        FROM dwh.stock
        GROUP BY itemid
    )

SELECT
    p.itemid AS "Код товара"
    , p.name_ax AS "Название товара"
    , p.brand_name_ax AS "Бренд"
    , p.category AS "Категория"
    , s.sales_qty_last_month AS "Сумма продаж в шт (прошлый месяц)"
    , s.sales_rub_last_month AS "Сумма продаж в руб (прошлый месяц)"
    , s.sales_rub_last_month / s.sales_qty_last_month AS "Средняя цена товара"
    , s.avg_qty_3m AS "Средний объем в шт (3 мес)"
    , st.actual_qty - s.avg_qty_3m AS "Разница (остаток - ср. продажи)"
FROM sales_data AS s
LEFT JOIN dwh.products AS p 
    ON s.itemid = p.itemid
LEFT JOIN stock_data AS st 
    ON s.itemid = st.itemid
"""

df = client.query_df(query_1)
display(df)

## ТОП-5 брендов по выручке

In [ ]:
query_2 = """
WITH
    toStartOfMonth(today()) AS start_curr_month
    , addMonths(start_curr_month, -1) AS start_last_month
    , addMonths(start_curr_month, -2) AS start_prev_month

    , sales_agg AS (
        SELECT
            itemid
            , sumIf(qty * price, date >= start_last_month AND date < start_curr_month) AS sales_rub_last_month
            , sumIf(qty * price, date >= start_prev_month AND date < start_last_month) AS sales_rub_prev_month
        FROM dwh.sales
        PREWHERE date >= start_prev_month 
            AND date < start_curr_month
        GROUP BY itemid
    )

SELECT
    p.brand_name_ax
    , sum(s.sales_rub_last_month) AS sales_rub_last_month
    , sum(s.sales_rub_prev_month) AS sales_rub_prev_month
    , sum(s.sales_rub_last_month) - sum(s.sales_rub_prev_month) AS sales_diff_rub
FROM sales_agg AS s
JOIN dwh.products AS p 
    ON s.itemid = p.itemid
GROUP BY p.brand_name_ax
ORDER BY sales_rub_last_month DESC
LIMIT 5
"""

df2 = client.query_df(query_2)
display(df2)